In [ ]:
import pandas as pd
import requests
import time

In [ ]:
# API Anfrage von einem Eintrag über Ozempic, speichere Ergebnis in Variable data
api_url ='https://api.fda.gov/drug/event.json' #API der FDA
params = {
    'search': 'patient.drug.medicinalproduct:"Ozempic"',
    'limit': 1
}
response =requests.get(api_url, params=params)
print(response.url)
data = response.json()

In [ ]:
# Hat Anfrage funktioniert?
response.status_code

In [ ]:
data

In [ ]:
# Speichere die Berichtsdaten des Patienten ohne Metadata in der Variable Bericht
bericht = data['results'][0]

In [ ]:
#Abfrage verschiedener Datenpunkte aus dem Bericht
print("Bericht-ID:", bericht["safetyreportid"])
print("Datum:", bericht["receivedate"])
print("Schwerwiegend:", bericht["serious"])
print("Geschlecht:", bericht["patient"].get("patientsex"))
print("Alter:", bericht["patient"].get("patientonsetage"))

print("Nebenwirkungen:")
for reaktion in bericht["patient"]["reaction"]:
    print(" -", reaktion["reactionmeddrapt"])

In [ ]:
response.close()

In [ ]:
# Welche Medikamente wurde eingenommen?
bericht["patient"]["drug"]

In [ ]:
# Welche Medikamente wurden eingenommen und welche Rolle hatten Sie (0 = Verdächtig die Nebenwirkund ausgelöst zu haben, 1 = Begleitmedikament)
for medikament in bericht["patient"]["drug"]:
    print(
        medikament.get("medicinalproduct"),
        "| Rolle:", medikament.get("drugcharacterization"),
        "| Anwendung:", medikament.get("drugindication")
    )

In [ ]:
# Welche Infos erhalten für über die eingenommenen Medikamente?
print(bericht["patient"]["drug"][0].keys())

In [ ]:
# API Anfrage zu einem Eintrag zu Semaglutide, speichere Ergebnis in Variable data
api_url ='https://api.fda.gov/drug/event.json' #API der FDA
params = {
    'search': 'patient.drug.openfda.generic_name:"semaglutide"',
    'limit': 1
}
response =requests.get(api_url, params=params)
print(response.url)
print(response.status_code)
data = response.json()
bericht = data['results'][0]
bericht

In [ ]:
for medikament in bericht["patient"]["drug"]:
    print(
        medikament.get("medicinalproduct"),
        "| Rolle:", medikament.get("drugcharacterization"),
        "| Anwendung:", medikament.get("drugindication")
    )

In [ ]:
# Welche Datenabfrage der API gibt mir den größten Datensatz? in den Meta daten wird nämlich immer die Gesamtzahl aller Berichte hierzu mitgeschickt
felder = [
    'patient.drug.openfda.generic_name:"semaglutide"', # FDA ergänzt nachträglich den Namen, kann aber manchmal auch nicht klappen
    'patient.drug.activesubstance.activesubstancename:"semaglutide"',# Sucht nach Berichten, wo semaglutide als Wirkstoff eingetragen wurde
    'patient.drug.medicinalproduct:"semaglutide"', # Sucht nach Berichten wo Semagltuif als Medikamentenname eingetragen wurde
]

for suche in felder:
    params = {'search': suche, 'limit': 1}
    response = requests.get(api_url, params=params)
    if response.status_code == 200:
        print(suche, '→', response.json()['meta']['results']['total'])
    else:
        print(suche, '→ Statuscode', response.status_code)

In [ ]:
felder = [
    'patient.drug.openfda.generic_name:"tirzepatide"',
    'patient.drug.activesubstance.activesubstancename:"tirzepatide"',
    'patient.drug.medicinalproduct:"tirzepatide"',
]

for suche in felder:
    params = {'search': suche, 'limit': 1}
    response = requests.get(api_url, params=params)
    if response.status_code == 200:
        print(suche, '→', response.json()['meta']['results']['total'])
    else:
        print(suche, '→ Statuscode', response.status_code)

In [ ]:
# Abfrage für Daten von Semaglutide für einen Monat. Benutze einen API-Key von der FDA damit ich mehr abfragen am Tag machen kann  (so gehen statt 1.000 Abfragen 120.000 am Tag)

# ---------- Importe ----------
import json                     # zum Speichern der Daten als JSON-Datei
import os                       # für Ordner und Dateipfade
import time                     # für Pausen zwischen den Abfragen
import requests                 # für die Abfragen an die API
from config import API_KEY # FDA-Schlüssel damit ich bis zu 120.000 Abfragen am Tag machen kann


# ---------- Variablen für Anfrage an API----------
active_ingredient = ["semaglutide", "tirzepatide"]
api_url ='https://api.fda.gov/drug/event.json' # API der FDA
start ='20250101' # Startdatum der Abfrage (YYYYMMDD)
end ='20250131' # Enddatum der Abfrage (YYYYMMDD), ertsmal nur für einen Monat, damit die Abfrage nicht zu groß wird
search ="patient.drug.activesubstance.activesubstancename:{} AND receivedate:[{} TO {}]".format(active_ingredient[0], start, end)

# ---------- Variablen für Schleife ----------
alle_berichte = [] # leere Liste in der alle Berichte von Wirkstoff gespeichert werden
skip = 0 # Startpunkt: Bei Bericht 0 anfangen und dann immer 1000 Berichte überspringen, bis alle Berichte abgefragt wurden
gesamt = 0 # Gesamtzahl der Treffer (wird bei der ersten Abfrage gesetzt)

# ---------- Download-Schleife zur Abfrage der Berichte ----------
while True: # Wiederholen, bis ein Break kommt
    params = {'search': search, 'limit': 1000, 'skip': skip, 'api_key': API_KEY} # Parameter für die Abfrage an die API
    response = requests.get(api_url, params=params) # Anfrage an die API

    if response.status_code != 200: # Wenn die Abfrage nicht erfolgreich war
        print('Stopp, Statuscode:', response.status_code) # Statuscode ausgeben
        break # Schleife beenden

    daten = response.json() # JSON-Daten aus der Antwort in einem Dictionary speichern
    alle_berichte.extend(daten['results']) # Neue Berichte an die Liste alle_berichte ranhängen
    gesamt = daten['meta']['results']['total'] # Gesamtzahl der Treffer aus den Metadaten speichern

    skip += 1000 # Nächste 1000 Berichte überspringen
    if skip >= gesamt: # Wenn alle Berichte abgefragt wurden
        break # Schleife beenden    
    time.sleep(0.5) # 0,5 Sekunden warten, um die API nicht zu überlasten

# ---------- Kontrolle ----------
print(active_ingredient[0], start, '-', end, ':', len(alle_berichte), 'von', gesamt, 'Berichten geladen')  # Vollständigkeit prüfen

# ---------- Speichern der Berichte als JSON-Datei ----------
dateiname = 'data/raw/{}_berichte_{}_bis_{}.json'.format(active_ingredient[0], start, end) # Dateiname, z. B. data/raw/semaglutide_20250101_bis_20250131.json
with open(dateiname, 'w') as f: # Datei zum Schreiben öffnen (wird danach automatisch geschlossen)
    json.dump(alle_berichte, f) # Berichte als JSON in die Datei schreiben

In [ ]:
# Abfrage für Daten von Semaglutide und Tirzepatide für das Jahr 2025

# ---------- Importe ----------
import calendar # für die Berechnung der letzten Tage eines Monats
import json # zum Speichern der Daten als JSON-Datei
import os # für Ordner und Dateipfade
import time  # für Pausen zwischen den Abfragen
import requests  # für die Abfragen an die API
from config import API_KEY # FDA-Schlüssel damit ich bis zu 120.000 Abfragen am Tag machen kann

# ---------- Variablen für Anfrage an API----------
active_ingredient = ["semaglutide", "tirzepatide"]
api_url ='https://api.fda.gov/drug/event.json' # API der FDA


# ---------- Download-Schleife zur Abfrage der Berichte ----------

for ingredient in active_ingredient: # Schleife über die Wirkstoffe
    for month in range(1, 13): # Schleife über die Monate Januar bis Dezember
        last_day = calendar.monthrange(2025, month)[1] # z. B. 31 für Januar, 28 für Februar
        start ='2025{:02d}01'.format(month) # Startdatum der Abfrage (YYYYMMDD), {:02d} macht aus 1 → 01
        end ='2025{:02d}{}'.format(month, last_day) # Enddatum der Abfrage (YYYYMMDD)
        search ="patient.drug.activesubstance.activesubstancename:{} AND receivedate:[{} TO {}]".format(ingredient, start, end)

        # ---------- Variablen für while-Schleife ----------
        alle_berichte = [] # leere Liste in der alle Berichte von Wirkstoff gespeichert werden
        skip = 0 # Startpunkt: Bei Bericht 0 anfangen und dann immer 1000 Berichte überspringen, bis alle Berichte abgefragt wurden
        gesamt = 0 # Gesamtzahl der Treffer (wird bei der ersten Abfrage gesetzt)

        while True: # Wiederholen, bis ein Break kommt
            params = {'search': search, 'limit': 1000, 'skip': skip, 'api_key': API_KEY} # Parameter für die Abfrage an die API
            response = requests.get(api_url, params=params) # Anfrage an die API

            if response.status_code != 200: # Wenn die Abfrage nicht erfolgreich war
                print('Stopp, Statuscode:', response.status_code) # Statuscode ausgeben
                break # Schleife beenden
            
            daten = response.json() # JSON-Daten aus der Antwort in einem Dictionary speichern
            alle_berichte.extend(daten['results']) # Neue Berichte an die Liste alle_berichte ranhängen
            gesamt = daten['meta']['results']['total'] # Gesamtzahl der Treffer aus den Metadaten speichern
            
            skip += 1000 # Nächste 1000 Berichte überspringen
            if skip >= gesamt: # Wenn alle Berichte abgefragt wurden
                break # Schleife beenden    
            time.sleep(0.5) # 0,5 Sekunden warten, um die API nicht zu überlasten
            
        # ---------- Kontrolle ----------
        print(ingredient, start, '-', end, ':', len(alle_berichte), 'von', gesamt, 'Berichten geladen')  # Vollständigkeit prüfen
        
        # ---------- Speichern der Berichte als JSON-Datei ----------
        dateiname = 'data/raw/{}_berichte_{}_bis_{}.json'.format(ingredient, start, end) # Dateiname, z. B. data/raw/semaglutide_berichte_20250101_bis_20250131.json
        with open(dateiname, 'w') as f: # Datei zum Schreiben öffnen (wird danach automatisch geschlossen)
            json.dump(alle_berichte, f) # Berichte als JSON in die Datei schreiben

        time.sleep(0.5) # kurze Pause vor dem nächsten Monat

In [ ]:
# Abfrage der Anzahl der Berichte pro Tag für Semaglutide und Tirzepatide über den gesamten Zeitraum (für eine Zeitreihe).

# ---------- Importe ----------
import requests              # für die Abfragen an die API
import pandas as pd          # für die Tabellen
from config import API_KEY   # FDA-Schlüssel aus der geheimen config.py

# ---------- Variablen für Anfrage an API ----------
active_ingredient = ["semaglutide", "tirzepatide"]   # Liste der Wirkstoffe
api_url = 'https://api.fda.gov/drug/event.json'      # API der FDA

# ---------- Abfrage ----------
zeitreihen = []   # leere Liste, in der die Tabellen beider Wirkstoffe gesammelt werden

for ingredient in active_ingredient:   # Schleife über die Wirkstoffe
    search = 'patient.drug.activesubstance.activesubstancename:"{}"'.format(ingredient)   # Suche nur nach Wirkstoff, ohne Zeitraum
    params = {'search': search, 'count': 'receivedate', 'api_key': API_KEY}   # count = Berichte pro Eingangsdatum zählen lassen
    response = requests.get(api_url, params=params)   # Anfrage an die API
    print(ingredient, 'Statuscode:', response.status_code)   # Kontrolle: 200 = alles ok

    df = pd.DataFrame(response.json()['results'])   # Ergebnis als Tabelle mit den Spalten "time" (Datum) und "count" (Anzahl)
    df['wirkstoff'] = ingredient   # neue Spalte mit dem Wirkstoffnamen
    zeitreihen.append(df)   # Tabelle an die Liste anhängen

# ---------- Zusammenfügen ----------
zeitreihe = pd.concat(zeitreihen)   # beide Tabellen untereinander zu einer Tabelle zusammenfügen

# ---------- Kontrolle ----------
print(zeitreihe.groupby('wirkstoff')['time'].agg(['min', 'max']))   # frühestes und spätestes Datum pro Wirkstoff
print(zeitreihe.groupby('wirkstoff')['count'].sum())   # Gesamtzahl der Berichte pro Wirkstoff
print(zeitreihe.head())   # erste 5 Zeilen der Tabelle anzeigen
print(zeitreihe.tail())   # letzte 5 Zeilen der Tabelle anzeigen

# ---------- Speichern ----------
zeitreihe.to_csv('data/raw/berichte_pro_tag.csv', index=False)   # als CSV-Datei speichern

In [ ]:
response.close()

In [ ]:
# Überprüfung Semaglutid Juli 2025
df = pd.read_csv('data/raw/berichte_pro_tag.csv', parse_dates=['time'])  # CSV-Datei einlesen und die Spalte "time" als Datum interpretieren
df.head()

In [ ]:
df.dtypes

In [ ]:
semaglutide_mask = df['wirkstoff'] == 'semaglutide'
mask_juni = (df['time'] >= '2025-06-01') & (df['time'] <= '2025-06-30')
df[semaglutide_mask & mask_juni].groupby('time')['count'].sum() # brauche eigentlich nicht zu gruppieren, aber mir gefällt die Ansicht besser

In [ ]:
semaglutide_mask = df['wirkstoff'] == 'semaglutide'
mask_juli = (df['time'] >= '2025-07-01') & (df['time'] <= '2025-07-31')
df[semaglutide_mask & mask_juli].groupby('time')['count'].sum()

In [ ]:
import json
with open('data/raw/semaglutide_berichte_20250701_bis_20250731.json', 'r') as f:
    berichte = json.load(f)

In [ ]:
len(berichte)

In [ ]:
df_juli = pd.DataFrame()   # leere Tabelle anlegen

for i in range(len(berichte)):   # für jeden Bericht (wie range(df.shape[0]))
    bericht_dict = berichte[i]   # ein Bericht als Dictionary (wie response_dict)
    quelle = bericht_dict.get('primarysource') or {}   # Infos zur meldenden Person, leer falls nicht vorhanden

    df_juli.loc[i, 'id'] = bericht_dict['safetyreportid']   # eindeutige Nummer des Berichts
    df_juli.loc[i, 'datum'] = bericht_dict['receivedate']    # Tag an dem FDA Bericht erhalten hat
    df_juli.loc[i, 'meldende'] = quelle.get('qualification') # Wer den Fall urspünglich gemeldet hat (z.b. 1= Arzt/Ärztin)
    df_juli.loc[i, 'land'] = bericht_dict.get('occurcountry') # Land, in dem die Nebenwirkungen aufgetreten sind
    df_juli.loc[i, 'berichtsart'] = bericht_dict.get('reporttype') # Woher der Bericht stammt (z.B. 1 = Spontanmeldung)
    df_juli.loc[i, 'firmennummer'] = bericht_dict.get('companynumb') # Fallnummer des Herstellers, der den Fall weitergeleitet hat

df_juli.head()

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])

df_juli[mask_ausreisser]['meldende'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['meldende'].value_counts(normalize=True)

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])

df_juli[mask_ausreisser]['land'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['land'].value_counts(normalize=True)

In [ ]:
mask_ausreisser = df_juli['datum'].isin(['20250715', '20250716', '20250718'])

df_juli[mask_ausreisser]['berichtsart'].value_counts(normalize=True)

In [ ]:
df_juli[~mask_ausreisser]['berichtsart'].value_counts(normalize=True)

In [ ]:
df_juli['firma'] = df_juli['firmennummer'].str.split('-').str[1]   # Firmenkürzel aus der Firmennummer herausziehen

df_juli[mask_ausreisser]['firma'].value_counts(normalize=True).head(10)    # Firmen an den Ausreißer-Tagen (Anteile, Top 10)

In [ ]:
df_juli[~mask_ausreisser]['firma'].value_counts(normalize=True).head(10)   # Firmen an den übrigen Tagen (Anteile, Top 10)

In [ ]:
import json
import os
import pandas as pd

ordner = 'data/raw'   # hier liegen die Rohdaten

# ---------- Alle JSON-Dateien im Ordner finden ----------
dateien = []   # leere Liste für die Dateinamen
for name in os.listdir(ordner):   # alle Dateien im Ordner durchgehen
    if name.endswith('.json'):    # nur JSON-Dateien (die CSV der Zeitreihe nicht)
        dateien.append(name)      # Dateinamen merken
print(len(dateien), 'Dateien gefunden')   # sollten 24 sein

In [ ]:
# ---------- Leere Listen für die drei Tabellen ----------
berichte_zeilen = []         # eine Zeile pro Bericht
nebenwirkungen_zeilen = []   # eine Zeile pro gemeldeter Nebenwirkung
medikamente_zeilen = []      # eine Zeile pro genanntem Medikament

# ---------- Durch alle Dateien und Berichte gehen ----------
for name in dateien:   # jede Monatsdatei
    abfrage = name.split('_')[0]   # aus 'semaglutide_berichte_...' wird 'semaglutide'
    with open(os.path.join(ordner, name), 'r') as f:   # Datei öffnen (os.path.join setzt Ordner + Name zusammen)
        berichte = json.load(f)   # Liste mit allen Berichten dieses Monats

    for bericht in berichte:   # jeden Bericht durchgehen
        bericht_id = bericht.get('safetyreportid')   # ID merken, sie verbindet später die drei Tabellen
        patient = bericht.get('patient') or {}       # Teil mit Patientendaten (leer, falls er fehlt)
        quelle = bericht.get('primarysource') or {}  # Teil mit der meldenden Person (leer, falls er fehlt)

        # --- Tabelle 1: Berichte ---
        berichte_zeilen.append({
            'id': bericht_id,
            'abfrage': abfrage,                                  # aus welchem Download der Bericht stammt
            'datum': bericht.get('receivedate'),                 # Eingangsdatum
            'schwerwiegend': bericht.get('serious'),             # 1 = ja, 2 = nein
            'geschlecht': patient.get('patientsex'),             # 0 = unbekannt, 1 = männlich, 2 = weiblich
            'alter': patient.get('patientonsetage'),             # Alter (Zahl)
            'alter_einheit': patient.get('patientonsetageunit'), # Einheit des Alters (Code)
            'meldende': quelle.get('qualification'),             # wer gemeldet hat
            'land': bericht.get('occurcountry'),                 # Land
            'berichtsart': bericht.get('reporttype'),            # Art des Berichts
            'firmennummer': bericht.get('companynumb'),          # Fallnummer des Herstellers
        })

# --- Tabelle 2: Nebenwirkungen ---
        for reaktion in patient.get('reaction', []):   # jede Nebenwirkung im Bericht (leere Liste, falls keine)
            nebenwirkungen_zeilen.append({
                'id': bericht_id,
                'abfrage': abfrage,                                 # aus welchem Download die Zeile stammt
                'nebenwirkung': reaktion.get('reactionmeddrapt'),   # Name der Nebenwirkung
            })

        # --- Tabelle 3: Medikamente ---
        for medikament in patient.get('drug', []):   # jedes Medikament im Bericht
            wirkstoff_info = medikament.get('activesubstance') or {}   # verschachtelter Teil mit dem Wirkstoff
            medikamente_zeilen.append({
                'id': bericht_id,
                'abfrage': abfrage,                                         # aus welchem Download die Zeile stammt
                'medikament': medikament.get('medicinalproduct'),          # eingetragener Name
                'wirkstoff': wirkstoff_info.get('activesubstancename'),     # Wirkstoff
                'rolle': medikament.get('drugcharacterization'),            # 1 = verdächtig, 2 = Begleitmedikation, 3 = Wechselwirkung
                'anwendung': medikament.get('drugindication'),              # wofür es genommen wurde
            })

# ---------- Aus den Listen Tabellen machen ----------
df_berichte = pd.DataFrame(berichte_zeilen)
df_nebenwirkungen = pd.DataFrame(nebenwirkungen_zeilen)
df_medikamente = pd.DataFrame(medikamente_zeilen)

print('Berichte:', df_berichte.shape)             # (Zeilen, Spalten)
print('Nebenwirkungen:', df_nebenwirkungen.shape)
print('Medikamente:', df_medikamente.shape)

# ---------- Speichern ----------
os.makedirs('data/processed', exist_ok=True)   # Ordner anlegen, falls es ihn noch nicht gibt
df_berichte.to_csv('data/processed/berichte.csv', index=False)
df_nebenwirkungen.to_csv('data/processed/nebenwirkungen.csv', index=False)
df_medikamente.to_csv('data/processed/medikamente.csv', index=False)

In [ ]:
# Berichte können doppelt vorkommen, wenn Tirzepatid und Semagltuig gegeben wurde... wie oft kommt das vor?
df_berichte['id'].duplicated().sum()   # wie oft kommt eine ID mehr als einmal vor?

In [ ]:
# ---------- Duplikate entfernen ----------
mask_doppelt = df_berichte['id'].duplicated()        # Maske: True, wenn die ID schon weiter oben vorkam
doppelte_ids = df_berichte[mask_doppelt]['id']       # aus diesen Zeilen nur die Spalte id nehmen

mask_berichte = df_berichte['id'].isin(doppelte_ids) & (df_berichte['abfrage'] == 'tirzepatide')   # doppelte Zeilen aus dem Tirzepatid-Download
df_berichte = df_berichte.drop(df_berichte[mask_berichte].index)   # diese Zeilen entfernen

mask_nebenwirkungen = df_nebenwirkungen['id'].isin(doppelte_ids) & (df_nebenwirkungen['abfrage'] == 'tirzepatide')
df_nebenwirkungen = df_nebenwirkungen.drop(df_nebenwirkungen[mask_nebenwirkungen].index)

mask_medikamente = df_medikamente['id'].isin(doppelte_ids) & (df_medikamente['abfrage'] == 'tirzepatide')
df_medikamente = df_medikamente.drop(df_medikamente[mask_medikamente].index)

# ---------- Kontrolle ----------
print('Berichte:', df_berichte.shape)             # (Zeilen, Spalten)
print('Nebenwirkungen:', df_nebenwirkungen.shape)
print('Medikamente:', df_medikamente.shape)
print('Doppelte IDs:', df_berichte['id'].duplicated().sum())   # sollte 0 sein

In [ ]:
# ---------- Spalte abfrage entfernen ----------
df_berichte = df_berichte.drop(columns='abfrage')             # Hilfsspalte entfernen
df_nebenwirkungen = df_nebenwirkungen.drop(columns='abfrage')
df_medikamente = df_medikamente.drop(columns='abfrage')

# ---------- Kontrolle ----------
print('Berichte:', df_berichte.shape)             
print('Nebenwirkungen:', df_nebenwirkungen.shape) 
print('Medikamente:', df_medikamente.shape)       

# ---------- Speichern ----------
df_berichte.to_csv('data/processed/berichte.csv', index=False)
df_nebenwirkungen.to_csv('data/processed/nebenwirkungen.csv', index=False)
df_medikamente.to_csv('data/processed/medikamente.csv', index=False)

In [ ]:
print('hallo')